# PROJECT TITLE: Understanding Revenue Drivers and Customer Purchasing Behaviour in an Online Retail Business

This notebook presents a case study analysis of transactional data from an online Retail Business.
The objective of this project is to explore transactional e-commerce data in order to identify revenue drivers, examine customer purchasing behaviour, and uncover temporal sales patterns.

The analysis follows the objectives outlined in the accompanying project proposal document.

## Case Background
An online retail company has accumulated a year's worth of transactional data from customer purchases.
Although overall sales figures are available, the management lacks clarity on what actually drives revenue, how customers differ in their purchasing behaviour, and whether meaningful trends exist over time.

This case study applies data analysis and statistical techniques to transform raw transaction data into insight that can support better business decisions.

## Business Questions
This analysis focuses on answering the following questions:

1. What factors are associated with higher revenue?
2. How does revenue change over time?
3. Do repeat customers generate more revenue than one-time customers

## Deliverable 2 - Data Collection & Cleaning

### Imports and setup for analysis

In [2]:
# import the necessary package 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Data Review 
The dataset contains transactional records from an online retail business between December 2010 and December 2011.
Each row represents a product purchased as part of an invoice.

Before any analysis, the dataset is loaded and inspected to understand its structure and contents.

### Load dataset

In [7]:
df_retail = pd.read_csv('Online Retail.csv')

# to read first 5 rows
df_retail.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01/12/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01/12/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01/12/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01/12/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01/12/2010 8:26,3.39,17850.0,United Kingdom


### Data Exploration
Understanding the size of the dataset, available columns, and data types helps identify potential data quality issues that may affect analysis.

In [8]:
# to check the size of the dataset
df_retail.shape

(541909, 8)

In [9]:
# to know what the columns entails
df_retail.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

In [10]:
# information on the columns
df_retail.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


### Data Cleaning and Imputation
The purpose is to

- check for missing values
- check for inconsistencies

In [13]:
# missing values 
df_retail.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Explanation: Missing values were observed to be in CustomerID and Description columns

### Remove Duplicates and Invalid Rows
The goal is to make sure every row represents a valid transaction. So I'm checking for:
- Duplicates entries

In [34]:
# Drop duplicates
df_retail = df_retail.drop_duplicates()

#### Data Imputation

For this analysis, the management observed that CustomerID and Description contains missing values, because they are categorical identifiers and not measurable quantities, their missing values mcannot be relably inferred using statistical methods, inputing them would introduce misleading information about the dataset. Therefore, rows with missing CustomerID and Description were removed because we cannot use statistical methods(mean/median) to guess a unique customerID or a specific product description. Removing these rows ensures that the analysis of customer behaviour and revenue trends is based on verified transactions.  

In [15]:
# handling missing values
df_retail = df_retail.dropna(subset=['CustomerID', 'Description'])

In [16]:
df_retail.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [18]:
# Remove invalid sales records
df_retail = df_retail[(df_retail['Quantity'] > 0) & (df_retail['UnitPrice'] > 0)]

Explanation for Removal of Invalid Transactions: The rows with zero or negative Quantity and UnitPrice were removed because they represent returns or data entry errors that do not reflect actual sales activity.

### Outlier Detection

A new feature, Revenue is created by multiplying Quantity and UnitPrice. This represents the value of each transaction and enables revenue-based analysis.

In [19]:
# feature engineering
df_retail['Revenue'] = np.multiply(df_retail['Quantity'], df_retail['UnitPrice'])

In [35]:
# calculate IQR
Q1 = df_retail['Revenue'].quantile(0.25)
Q3 = df_retail['Revenue'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [26]:
# to check the number of outliers
outliers = df_retail[df_retail['Revenue'] > upper_bound]
print(f'Number of outliers: {len(outliers)}')
outliers.head(10)

Number of outliers: 31241


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,01/12/2010 8:34,1.69,13047.0,United Kingdom,54.08
26,536370,22728,ALARM CLOCK BAKELIKE PINK,24,01/12/2010 8:45,3.75,12583.0,France,90.00
27,536370,22727,ALARM CLOCK BAKELIKE RED,24,01/12/2010 8:45,3.75,12583.0,France,90.00
28,536370,22726,ALARM CLOCK BAKELIKE GREEN,12,01/12/2010 8:45,3.75,12583.0,France,45.00
33,536370,21035,SET/2 RED RETROSPOT TEA TOWELS,18,01/12/2010 8:45,2.95,12583.0,France,53.10
34,536370,22326,ROUND SNACK BOXES SET OF4 WOODLAND,24,01/12/2010 8:45,2.95,12583.0,France,70.80
35,536370,22629,SPACEBOY LUNCH BOX,24,01/12/2010 8:45,1.95,12583.0,France,46.80
36,536370,22659,LUNCH BOX I LOVE LONDON,24,01/12/2010 8:45,1.95,12583.0,France,46.80
37,536370,22631,CIRCUS PARADE LUNCH BOX,24,01/12/2010 8:45,1.95,12583.0,France,46.80
40,536370,22900,SET 2 TEA TOWELS I LOVE LONDON,24,01/12/2010 8:45,2.95,12583.0,France,70.80


Explanation: The interquatile range (IQR) method was use to identify extreme transaction values in Revenue

In [36]:
# capping the outliers
df_retail['Revenue'] = np.clip(df_retail['Revenue'], lower_bound, upper_bound)

Explanation: Although extreme revenue were identified, making the distributions right skewed, these transections may represent legitimate high-value purchases. Therefore, instead of removing them, values were cappped at the IQR bounds to redations auce their influence while retaining meaningful observations and insights.

### Feature Engineering
This is done in order to enable time based analysis and exploration

In [46]:
df_retail['InvoiceDate'] = pd.to_datetime(df_retail['InvoiceDate'])

In [32]:
# create time features
df_retail['Year'] = df_retail['InvoiceDate'].dt.year
df_retail['Month'] = df_retail['InvoiceDate'].dt.month

Explanation: The InvoiceDate column was converted to datetime format, year and month were created from the InvoiceDate column to support time-based analysis. The features allows revenue to be analyzed across different months and across years, helping to identify trends, growth patterns, and possible seasonal effects in customer purchasing behaviour. 

In [39]:
# check for the validity
df_retail.isnull().sum()
df_retail.info()

<class 'pandas.core.frame.DataFrame'>
Index: 392692 entries, 0 to 541908
Data columns (total 11 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    392692 non-null  object        
 1   StockCode    392692 non-null  object        
 2   Description  392692 non-null  object        
 3   Quantity     392692 non-null  int64         
 4   InvoiceDate  392692 non-null  datetime64[ns]
 5   UnitPrice    392692 non-null  float64       
 6   CustomerID   392692 non-null  float64       
 7   Country      392692 non-null  object        
 8   Revenue      392692 non-null  float64       
 9   Year         392692 non-null  int32         
 10  Month        392692 non-null  int32         
dtypes: datetime64[ns](1), float64(3), int32(2), int64(1), object(4)
memory usage: 33.0+ MB


Explanation: The final validity checks confirmed the dataset is clean and consistent

In [42]:
# Save Clean Dataset
df_retail.to_csv("preprocessed_retail_data.csv", index=False)

#### Explanation
Missing values were identified primarily in CustomerID and Description. Because these variables are categorical identifiers and cannot be reliably estimated using statistical measures, rows with missing values in these columns were removed. Duplicate records were dropped to prevent double-counting of transactions. Rows with zero or negative Quantity and UnitPrice were removed as they represent returns or data entry errors. Outliers in Revenue were capped using the IQR method rather than removed to preserve valid high-value purchases. These steps resulted in a clean, consistent dataset suitable for analysis.
